# NaijaABSA: collecting Nigerian Pidgin app reviews from Google Play

Author: Juliet Chinenye Duru

This notebook collects public Google Play reviews from Nigerian apps in ten sectors, keeps the reviews written in Nigerian Pidgin or code-switched Pidgin, and removes foreign-language reviews.

**Run order:** Step 1 (collect, rerun if Colab disconnects) → Step 2 (filter for Pidgin) → Step 3 (remove foreign-language reviews).

All files are saved to `MyDrive/NaijaABSA` in your own Google Drive. Review text is never saved to this repository.

## Step 1: Collect reviews

In [ ]:
# I collect Google Play reviews from Nigerian apps in ten sectors and save each app to my Google Drive so nothing is lost

!pip install -q google-play-scraper
import os, time
import pandas as pd
from google_play_scraper import reviews, Sort
from google.colab import drive

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/NaijaABSA/raw_reviews"
os.makedirs(SAVE_DIR, exist_ok=True)

APPS = {
    "telecoms":      ["ng.mtn.nextgen", "net.one97.selfcare.globacom", "com.airtel.africa.selfcare"],
    "banking":       ["com.firstbank.firstmobile", "com.moniepoint.business", "com.moniepoint.personal",
                      "team.opay.pay", "com.kudabank.app", "com.transsnet.palmpay",
                      "com.gtbank.gtworldv1", "com.accessbank.nextgen", "com.ubanquity.redd.uba"],
    "electricity":   ["com.vatebrapay.fcmbenergy", "com.ionicframework.buypowerapp334891", "com.powerlight.ng"],
    "ecommerce":     ["com.jumia.android", "ng.jiji.app", "com.konga.androida"],
    "ride-hailing":  ["ee.mtakso.client", "sinet.startup.inDriver", "com.ubercab"],
    "food-delivery": ["com.chowdeck.com", "com.glovo"],
    "education":     ["com.ulesson"],
    "travel":        ["com.flyairpeace.app.airpeace", "com.gigm", "com.wakanow.android"],
    "government":    ["com.nimcmobile"],
    "pay-tv":        ["com.dstv.mydstv", "com.dstv.mygotv", "com.multichoice.gotv", "com.dstvmobile.android"],
}
MAX_PAGES = 150

for domain, ids in APPS.items():
    for app_id in ids:
        out = f"{SAVE_DIR}/{app_id}.csv"
        if os.path.exists(out):
            print(f"Already saved, skipping: {app_id}")
            continue
        rows, token = [], None
        for _ in range(MAX_PAGES):
            try:
                batch, token = reviews(app_id, lang="en", country="ng", sort=Sort.NEWEST,
                                       count=200, continuation_token=token)
            except Exception as e:
                print("Problem with", app_id, "-", e); break
            rows += [{"review_id": r["reviewId"], "app_id": app_id, "domain": domain,
                      "text": r["content"], "date": str(r["at"])} for r in batch]
            if not batch or token is None:
                break
            time.sleep(2)
        pd.DataFrame(rows, columns=["review_id", "app_id", "domain", "text", "date"]).to_csv(out, index=False)
        print(f"{domain:14s} {app_id:40s} {len(rows)} reviews saved")


## Step 2: Clean and keep Pidgin reviews

In [ ]:
# I combine all my saved reviews, remove personal details, keep the Pidgin ones, and count them per sector

import glob, re
import pandas as pd
from google.colab import drive, files

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/NaijaABSA/raw_reviews"

df = pd.concat([pd.read_csv(f) for f in glob.glob(f"{SAVE_DIR}/*.csv")], ignore_index=True)
df = df.dropna(subset=["text"]).drop_duplicates("review_id").drop_duplicates("text")
print("All reviews collected:", len(df))

def scrub(t):
    t = re.sub(r"\S+@\S+", "[EMAIL]", str(t))
    t = re.sub(r"(\+?234|0)[789][01]\d{8}", "[PHONE]", t)
    t = re.sub(r"\b\d{10,}\b", "[NUMBER]", t)
    t = re.sub(r"@\w+", "[USER]", t)
    t = re.sub(r"https?://\S+", "[URL]", t)
    return t.strip()

df["text"] = df["text"].map(scrub)
df = df[df["text"].str.split().str.len().between(5, 80)]

POWER = r"\b(?:nepa|phcn|light|token|meter|prepaid meter|electricity|disco|ikedc|ekedc|aedc|ibedc|eedc|units?)\b"
is_power = df["text"].str.lower().str.contains(POWER) & df["domain"].isin(["banking", "telecoms"])
df.loc[is_power, "domain"] = "electricity"

MARKERS = ["dey", "wey", "abeg", "no be", "una", "wetin", "sabi", "oga", "wahala", "comot",
           "abi", "sha", "shey", "e don", "don dey", "na im", "nawa", "chop", "o jare", "biko",
           "dem", "wan", "make una", "e no"]
pattern = r"\b(?:" + "|".join(re.escape(m) for m in MARKERS) + r")\b"
pidgin = df[df["text"].str.lower().str.contains(pattern)]

print("Pidgin reviews:", len(pidgin))
print(pidgin.domain.value_counts())

pidgin.to_json("/content/drive/MyDrive/NaijaABSA/pidgin_pool.jsonl", orient="records", lines=True)
pidgin.to_json("pidgin_pool.jsonl", orient="records", lines=True)
files.download("pidgin_pool.jsonl")


## Step 3: Remove foreign-language reviews (final 2,351)

In [ ]:
# I remove non-Nigerian, foreign-language reviews from my Pidgin pool and save the final set of Nigerian Pidgin reviews

!pip install -q langdetect==1.0.9
import re
import pandas as pd
from langdetect import detect_langs, DetectorFactory
from google.colab import drive, files

# I fix the langdetect version to 1.0.9 and set the random seed to 0, so my cleaning step gives exactly the same 2,351 reviews every time I run it.
DetectorFactory.seed = 0
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/NaijaABSA"

df = pd.read_json(f"{BASE}/pidgin_pool.jsonl", lines=True)
print("Pidgin pool before cleaning:", len(df))

STRONG = ["dey", "wey", "abeg", "no be", "wetin", "sabi", "oga", "wahala", "comot", "abi", "sha",
          "shey", "e don", "don dey", "na im", "nawa", "chop", "o jare", "biko", "make una", "e no",
          "una don", "una dey", "una go", "una no"]
strong_pattern = r"\b(?:" + "|".join(re.escape(m) for m in STRONG) + r")\b"
low = df["text"].str.lower()

def main_language(text):
    try:
        return detect_langs(text)[0].lang
    except Exception:
        return "unknown"

df["language_detected"] = df["text"].map(main_language)
FOREIGN = ["es", "pt", "it", "ca", "fr", "de", "ro", "gl"]
rule_1 = df["language_detected"].isin(FOREIGN) & (low.str.count(strong_pattern) == 0)

PIDGIN_WITHOUT_SHA = [m for m in STRONG if m != "sha"] + ["una"]
pattern_without_sha = r"\b(?:" + "|".join(re.escape(m) for m in PIDGIN_WITHOUT_SHA) + r")\b"
rule_2 = low.str.contains(r"ma\s*sha\.?\s*allah") & (low.str.count(pattern_without_sha) == 0)

CHECKED_BY_HAND = [
    "d11f1915-da52-4f3a-ad00-272a5a6956b6",  # Swahili
    "ce161e26-24da-4843-9580-40772bfe694a",  # Swahili
    "7628565a-62dd-47a1-98ed-6a12e44c53a8",  # Somali
    "8990f431-a33b-414b-9893-4e961605ab24",  # Estonian
    "0f0eb3a3-be18-41c1-a1ef-b632dadd798f",  # Malay
    "f973307f-190d-4b59-a9bd-839411f2740b",  # Hindi/Urdu
    "1e80e671-401a-4c15-8cd6-028175115d44",  # Urdu
    "d2c93fe6-e5d7-4c09-9aac-31d3a58208e3",  # Khasi
    "04d3398b-f4a0-4a5a-8c18-037db0058745",  # Croatian
]
rule_3 = df["review_id"].isin(CHECKED_BY_HAND)

removed = df[rule_1 | rule_2 | rule_3]
final = df[~(rule_1 | rule_2 | rule_3)].drop(columns="language_detected")

print("Removed as foreign-language:", len(removed))
print("Final Nigerian Pidgin reviews:", len(final))
print(final["domain"].value_counts())

final.to_json(f"{BASE}/nigerian_pidgin_reviews.jsonl", orient="records", lines=True)
removed.to_csv(f"{BASE}/removed_foreign_reviews.csv", index=False)
final.to_csv("nigerian_pidgin_reviews.csv", index=False, encoding="utf-8-sig")
files.download("nigerian_pidgin_reviews.csv")
